In [64]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import pandas as pd
import geopandas as gpd

from matplotlib import pyplot as plt
from shapely.geometry import Point, Polygon
from ipywidgets import interact_manual as interact, FloatSlider, IntRangeSlider, IntSlider
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

In [48]:
def cluster_shipwrecks(n_clusters: int, df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    features = df[["km_to_coast", "km_to_neighbor"]]
    
    iso = IsolationForest(contamination=0.01, random_state=42)
    is_inlier = iso.fit_predict(features)
    
    df = df[is_inlier == 1].copy()
    
    features = df[["km_to_coast", "km_to_neighbor"]]

    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    df["cluster"] = kmeans.fit_predict(scaled_features)
    return df

In [49]:
wrecks_df = pd.read_excel("data/shipwrecks_with_distance_to_neighbor.xlsx")
wrecks_df = cluster_shipwrecks(2, wrecks_df)

In [50]:
wrecks_gdf = gpd.GeoDataFrame(
    wrecks_df,
    geometry=gpd.points_from_xy(wrecks_df.lon, wrecks_df.lat),
    crs="EPSG:4326"
)

In [51]:
countries = gpd.read_file("data/natural_earth/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp")

In [52]:
bermuda_triangle = gpd.GeoDataFrame(
    {"name": ["Bermuda Triangle"]},
    geometry=[Polygon([(-80.19, 25.774), (-66.105, 18.466), (-64.75, 32.3078)])], # NOTE: (lon, lat) pairs
    crs="EPSG:4326",
)

In [53]:
def proportion_grid(xx: np.ndarray, yy: np.ndarray, years: tuple[int, int], gdf: gpd.GeoDataFrame, k: int = 10, max_dist: float = 2.) -> np.ndarray:
    wreck_coords = np.column_stack((gdf.geometry.x, gdf.geometry.y))
    wreck_tree = cKDTree(wreck_coords)
    w_years = gdf.date.dt.year.values

    n_wrecks = len(gdf)
    if k > n_wrecks:
        k = n_wrecks
    
    grid_points = np.column_stack((xx.ravel(), yy.ravel()))
    dists, indices = wreck_tree.query(grid_points, k=k)
    
    if k == 1:
        dists = dists.reshape(-1, 1)
        indices = indices.reshape(-1, 1)

    nearest_years = w_years[indices]

    t_min, t_max = years
    
    dist_mask = dists <= max_dist
    time_mask = (t_min <= nearest_years) & (nearest_years <= t_max)
    
    in_window = (dist_mask & time_mask).sum(axis=1)
    nn = dist_mask.sum(axis=1)

    prop = np.divide(
        in_window, 
        nn, 
        out=np.zeros_like(in_window, dtype=float), 
        where=0 < nn
    )

    return prop.reshape(xx.shape)

In [54]:
def draw_worldmap(ax: plt.Axes, gdf: gpd.GeoDataFrame, lat: float, lon: float, zoom: float, wrecks: bool, years: tuple[int, int], triangle: bool, overlay: bool, smooth: bool, k_nearest: int, max_dist: float, resolution: int) -> None:
    margin = 100. * (1 / zoom)
    x_min = lon - margin; x_max = lon + margin
    y_min = lat - margin; y_max = lat + margin

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_axis_off()

    if overlay:
        grid_x, grid_y = np.meshgrid(
            np.linspace(x_min, x_max, resolution),
            np.linspace(y_min, y_max, resolution)
        )
        prop_grid = proportion_grid(grid_x, grid_y, years, gdf, k=k_nearest, max_dist=max_dist)
        ax.imshow(
            prop_grid,
            extent=(x_min, x_max, y_min, y_max),
            origin="lower",
            cmap="coolwarm",
            alpha=0.6,
            interpolation="quadric" if smooth else "none"
        )

    countries.plot(ax=ax, facecolor="teal", edgecolor="black", linewidth=0.1)

    if triangle:
        bermuda_triangle.boundary.plot(ax=ax, color="black", linewidth=0.5 * zoom)

    if wrecks:
        min_year, max_year = years
        pts = gdf
        pts = pts.cx[x_min:x_max, y_min:y_max]
        pts = pts[(pts.date.dt.year >= min_year) & (pts.date.dt.year <= max_year)]
        pts.plot(
            ax=ax, 
            column="cluster",
            cmap="viridis", 
            markersize=5 * zoom, 
            marker="o",
            alpha=.6,
        )


In [55]:
def draw_clusters(ax: plt.Axes, df: pd.DataFrame) -> None:
    sc = ax.scatter(
        df["km_to_coast"], 
        df["km_to_neighbor"], 
        c=df["cluster"], 
        cmap="viridis", 
        alpha=0.5,
        s=10
    )
    ax.set_xlabel("distance to coast (km)")
    ax.set_ylabel("distance to nearest neighbor (km)")
    plt.colorbar(sc, label="cluster", ax=ax)
    ax.grid(True, linestyle="--", alpha=0.3)

In [56]:
def draw_timeline(ax: plt.Axes, df: pd.DataFrame, years: tuple[int, int], in_bermuda: bool, in_total: bool, in_outside: bool, normalize: bool) -> None:
    from_year, to_year = years
    lb = lambda year: year <= df.date.dt.year
    ub = lambda year: df.date.dt.year < year
    nm = lambda year: np.sum(lb(year) & ub(year + 1))

    y_total = np.array([nm(year) for year in range(from_year, to_year + 1)])
    y_bermuda = np.array([np.sum(lb(year) & ub(year + 1) & df.in_bermuda) for year in range(from_year, to_year + 1)])
    y_not_bermuda = y_total - y_bermuda

    if normalize:
        y_total = y_total / np.sum(y_total)
        y_bermuda = y_bermuda / np.sum(y_bermuda)
        y_not_bermuda = y_not_bermuda / np.sum(y_not_bermuda)

    x = np.arange(from_year, to_year + 1)

    if in_total:
        ax.plot(x, y_total, label="total wrecks" if not normalize else "% of total wrecks")
    if in_bermuda:
        ax.plot(x, y_bermuda, label="wrecks in Bermuda Triangle" if not normalize else "% of wrecks in Bermuda Triangle")
    if in_outside:
        ax.plot(x, y_not_bermuda, label="wrecks outside Bermuda Triangle" if not normalize else "% of wrecks outside Bermuda Triangle")
    ax.set_xlabel("year")
    ax.set_ylabel("# of Wrecks" if not normalize else "% of Wrecks")
    ax.set_title(f"wrecks per year normalized by subpopulation" if normalize else "wrecks per year per subpopulation")
    ax.legend()

In [67]:
def draw_timeline(ax: plt.Axes, df: pd.DataFrame, years: tuple[int, int], in_bermuda: bool, in_total: bool, in_outside: bool, normalize: bool) -> None:
    from_year, to_year = years
    x = np.arange(from_year, to_year + 1)

    mask_year = (df.date.dt.year >= from_year) & (df.date.dt.year <= to_year)
    df_view = df[mask_year].copy()
    
    allc = sorted(df["cluster"].unique())
    cmap = plt.get_cmap("viridis")
    norm = plt.Normalize(vmin=min(allc), vmax=max(allc))
    c2c_map = {c: cmap(norm(c)) for c in allc}
    cc = [c2c_map[c] for c in allc]

    def plot_with_clusters(mask: np.typing.ArrayLike, color: str, label_base: str) -> None:
        sub_df = df[mask]
        if sub_df.empty:
            return
            
        counts = sub_df.groupby([sub_df.date.dt.year, "cluster"]).size().unstack(fill_value=0)
        counts = counts.reindex(x, fill_value=0)
        
        for c in allc:
            if c not in counts.columns:
                counts[c] = 0
        counts = counts[allc]
        
        y_stack = counts.values.T
        y_total = y_stack.sum(axis=0)
        
        if normalize:
            total_sum = y_total.sum()
            if total_sum > 0:
                y_stack = y_stack / total_sum
                y_total = y_total / total_sum
                
        ax.stackplot(x, y_stack, colors=cc, alpha=0.3)
        label = label_base if not normalize else f"% of {label_base}"
        ax.plot(x, y_total, color=color, label=label)

    if in_total:
        plot_with_clusters(np.ones(len(df), dtype=bool), "gray", "total wrecks")
    if in_bermuda and "in_bermuda" in df.columns:
        plot_with_clusters(df.in_bermuda, "red", "wrecks in Bermuda Triangle")
    if in_outside and "in_bermuda" in df.columns:
        plot_with_clusters(~df.in_bermuda, "blue", "wrecks outside Bermuda Triangle")

    ax.set_xlabel("year")
    ax.set_ylabel("# of Wrecks" if not normalize else "% of Wrecks")
    ax.set_title(f"wrecks per year normalized by total subpopulation over all years" if normalize else "wrecks per year per subpopulation count")
    
    handles = []
    if in_total:
        handles.append(Line2D([0], [0], color="gray", lw=1.5, label="total wrecks" if not normalize else "% of total wrecks"))
    if in_outside and "in_bermuda" in df_view.columns:
        handles.append(Line2D([0], [0], color="blue", lw=1.5, label="wrecks outside Bermuda Triangle" if not normalize else "% of wrecks outside Bermuda Triangle"))
    if in_bermuda and "in_bermuda" in df_view.columns:
        handles.append(Line2D([0], [0], color="red", lw=1.5, label="wrecks in Bermuda Triangle" if not normalize else "% of wrecks in Bermuda Triangle"))
    for c in allc:
        handles.append(Patch(facecolor=c2c_map[c], edgecolor='none', alpha=0.3, label=f"Cluster {c}"))
    ax.legend(handles=handles, loc='upper left')

In [70]:
def inspect(lat: float, lon: float, zoom: float, wrecks: bool, years: tuple[int, int], triangle: bool, overlay: bool, smooth: bool, k_nearest: int, max_dist: float, resolution: int, n_clusters: int, timeline: str) -> None:
    df = cluster_shipwrecks(n_clusters, wrecks_df)
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.lon, df.lat),
        crs="EPSG:4326"
    )
    
    fig = plt.figure(figsize=(24, 18))
    gs = fig.add_gridspec(2, 2, height_ratios=[2, 1])
    ax_worldmap = fig.add_subplot(gs[0, 0])
    ax_clusters = fig.add_subplot(gs[0, 1])
    ax_timeline = fig.add_subplot(gs[1, :])

    draw_worldmap(
        ax_worldmap,
        gdf, 
        lat, lon, zoom, 
        wrecks, years, triangle, overlay, smooth, 
        k_nearest, max_dist, resolution
    )
    draw_clusters(ax_clusters, df)
    draw_timeline(
        ax_timeline,
        df,
        (df.date.dt.year.min(), df.date.dt.year.max()),
        in_bermuda=(timeline == "only in Bermuda" or timeline == "both in and out of Bermuda"),
        in_total=False,
        in_outside=(timeline == "only outside Bermuda" or timeline == "both in and out of Bermuda"),
        normalize=True
    )
    plt.show()

min_years = wrecks_df.date.dt.year.min()
max_years = wrecks_df.date.dt.year.max()
interact(
    inspect, 
    lat=FloatSlider(min=-90., max=+90., step=5., value=+25.), 
    lon=FloatSlider(min=-180., max=+180., step=5., value=-75.), 
    zoom=FloatSlider(min=0.01, max=+10., step=0.1, value=4.71),
    wrecks=True,
    years=IntRangeSlider(min=min_years, max=max_years, step=1, value=(2010, max_years)),
    triangle=True,
    overlay=True,
    smooth=False,
    k_nearest=IntSlider(min=1, max=50, step=1, value=20),
    max_dist=FloatSlider(min=0.1, max=10., step=0.1, value=2.),
    resolution=IntSlider(min=10, max=100, step=10, value=50),
    n_clusters=IntSlider(min=2, max=10, step=1, value=3),
    timeline=["both in and out of Bermuda", "only in Bermuda", "only outside Bermuda"]
);

interactive(children=(FloatSlider(value=25.0, description='lat', max=90.0, min=-90.0, step=5.0), FloatSlider(v…